# Team Overbye Weather Data API — Demo

This notebook walks through every major operation available through the `TeamOverbyeWeather` Python SDK and the raw REST API.

**Backend URL:** `https://weatherdatagui-production.up.railway.app`  
**Portal:** `https://weather-data-gui.pages.dev`

---

## Contents
1. [Install & Import](#1-install--import)
2. [Connect & Health Check](#2-connect--health-check)
3. [Browse the Catalog](#3-browse-the-catalog)
4. [ERA5 — List & Download](#4-era5--list--download)
5. [HRRR History — List & Download](#5-hrrr-history--list--download)
6. [HRRR Forecast — List & Download](#6-hrrr-forecast--list--download)
7. [NOAA / GFS — List & Download](#7-noaa--gfs--list--download)
8. [Raw REST API (no SDK)](#8-raw-rest-api-no-sdk)

9. [Region Catalog — Browse States & ISO Zones](#9-region-catalog--browse-available-states--iso-zones)
10. [Region Crop — SDK](#10-region-crop--sdk)
11. [Region Crop — Raw REST API](#11-region-crop--raw-rest-api)

---
## 1. Install & Import

In [3]:
%pip install TeamOverbyeWeather --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from TeamOverbyeWeather import WeatherClient

# Points to the production Railway backend by default.
# For local testing: WeatherClient(base_url="http://localhost:8000")
client = WeatherClient()

print("SDK ready. Base URL:", client._base_url)

SDK ready. Base URL: https://weatherdatagui-production.up.railway.app


---
## 2. Connect & Health Check

Two quick endpoints to confirm the backend is up and that all data pipelines are healthy.

In [5]:
import requests

API = client._base_url

health = requests.get(f"{API}/api/health", timeout=10).json()
print("Health:", health)

Health: {'status': 'ok'}


In [6]:
# Pipeline status: 'ok' | 'error' | 'unknown'
status = client.status()

for pipeline, state in status.items():
    icon = "✅" if state == "ok" else ("❌" if state == "error" else "❓")
    print(f"  {icon}  {pipeline:<22} {state}")

  ❓  noaa                   unknown
  ❓  hrrr_forecast          unknown
  ❓  hrrr_history           unknown
  ❓  era5                   unknown


---
## 3. Browse the Catalog

The catalog lists every available file grouped by source.  
It is cached on the server for 30 minutes; use `/api/catalog/refresh` to force a rebuild.

In [7]:
catalog = client.catalog()
print("Catalog keys:", list(catalog.keys()))

Catalog keys: ['hrrr_forecast', 'hrrr_history', 'hrrr_history_current', 'hrrr_history_archive', 'noaa_forecast', 'noaa_forecast_recent', 'noaa_forecast_archive', 'era5_na', 'era5_tx']


In [8]:
# Summary: how many items are available per source
for key, data in catalog.items():
    if not isinstance(data, dict):
        continue
    # Each source uses a different list key: quarters / months / days / cycles
    for list_key in ("quarters", "months", "days", "cycles"):
        items = data.get(list_key, [])
        if items:
            print(f"  {key:<28} {len(items):>4} {list_key}   (latest: {items[0]})")
            break

  hrrr_forecast                 172 cycles   (latest: 2026-05-22T12Z)
  hrrr_history                  136 months   (latest: 2026-05)
  hrrr_history_current           43 days   (latest: 2026-05-21)
  hrrr_history_archive          134 months   (latest: 2026-04)
  noaa_forecast                2022 cycles   (latest: 2026-05-22T12Z)
  noaa_forecast_recent         2022 cycles   (latest: 2026-05-22T12Z)
  noaa_forecast_archive        1844 cycles   (latest: 2026-04-08T00Z)
  era5_na                       346 quarters   (latest: 2026-Q2)


---
## 4. ERA5 — List & Download

ERA5 reanalysis is archived quarterly. Two regions are available:
- `"north_america"` — 0.25° grid, full CONUS + Canada
- `"texas"` — same resolution, Texas subset

In [9]:
# List available quarters for North America
na_quarters = client.era5.list_quarters(region="north_america")
print(f"North America — {len(na_quarters)} quarters available")
print("  Latest 5:", na_quarters[:5])

North America — 346 quarters available
  Latest 5: ['2026-Q2', '2026-Q1', '2025-Q4', '2025-Q3', '2025-Q2']


In [10]:
# Texas subset
tx_quarters = client.era5.list_quarters(region="texas")
print(f"Texas — {len(tx_quarters)} quarters available")
print("  Latest 5:", tx_quarters[:5])

Texas — 0 quarters available
  Latest 5: []


In [11]:
# Download the most recent ERA5 North America quarter
# Files are saved to ./data/  (created automatically)

if na_quarters:
    latest_q = na_quarters[0]
    print(f"Downloading ERA5 NA {latest_q} ...")
    paths = client.era5.download(
        quarters=[latest_q],
        region="north_america",
        dest="./data/era5",
    )
    print("Saved to:", paths[0])
else:
    print("No ERA5 quarters available.")

ERA5_era5_na_2026-Q2.pww: 100%|██████████| 206M/206M [00:34<00:00, 6.31MB/s] 

Saved to: data\era5\ERA5_era5_na_2026-Q2.pww


In [ ]:
# Download a range of quarters (bundled as a single ZIP by the server)
# Example: latest 3 quarters

if len(na_quarters) >= 3:
    range_quarters = na_quarters[:3]
    print(f"Downloading ERA5 NA {range_quarters} as bundle...")
    paths = client.era5.download(
        quarters=range_quarters,
        region="north_america",
        dest="./data/era5",
    )
    print("Saved to:", paths[0])

---
## 5. HRRR History — List & Download

HRRR historical data comes in two sub-catalogs:
- **Current year** (`hrrr_history_current`) — individual daily files
- **Archive** (`hrrr_history_archive`) — monthly bundles for past years

The SDK's `list_months()` / `download_history()` use the combined `hrrr_history` view.

In [ ]:
months = client.hrrr.list_months()
print(f"HRRR history — {len(months)} months available")
print("  Latest 6:", months[:6])

In [ ]:
# Individual daily files (current year)
days = catalog.get("hrrr_history_current", {}).get("days", [])
print(f"HRRR current-year daily files — {len(days)} days")
print("  Latest 5:", days[:5])

In [ ]:
# Download a single HRRR history month
if months:
    latest_month = months[0]
    print(f"Downloading HRRR history {latest_month} ...")
    paths = client.hrrr.download_history(
        months=[latest_month],
        dest="./data/hrrr",
    )
    print("Saved to:", paths[0])
else:
    print("No HRRR history months available.")

In [ ]:
# Download multiple months as a bundle
if len(months) >= 2:
    selected = months[:2]
    print(f"Downloading HRRR history {selected} as bundle...")
    paths = client.hrrr.download_history(months=selected, dest="./data/hrrr")
    print("Saved to:", paths[0])

---
## 6. HRRR Forecast — List & Download

48-hour HRRR forecasts, updated 4× per day (00Z, 06Z, 12Z, 18Z).  
Cycle keys use the format `YYYY-MM-DDTHHZ`.

In [ ]:
forecast_cycles = client.hrrr.list_forecast_cycles()
print(f"HRRR forecast — {len(forecast_cycles)} cycles available")
print("  Latest 5:", forecast_cycles[:5])

In [ ]:
# Download the most recent HRRR forecast in one call
print("Downloading latest HRRR forecast...")
path = client.hrrr.download_latest_forecast(dest="./data/hrrr_forecast")
print("Saved to:", path)

In [ ]:
# Download a specific cycle by key
if forecast_cycles:
    cycle = forecast_cycles[0]
    print(f"Downloading HRRR forecast cycle {cycle} ...")
    paths = client.hrrr.download_forecast(cycles=[cycle], dest="./data/hrrr_forecast")
    print("Saved to:", paths[0])

---
## 7. NOAA / GFS — List & Download

GFS 384-hour forecasts at 0.25°, covering North America.  
Files are in `.pww` format (Team Overbye binary). Updated every 6 hours.

Two sub-catalogs:
- **Recent** (`noaa_forecast_recent`) — last ~16 days
- **Archive** (`noaa_forecast_archive`) — older cycles

In [ ]:
noaa_cycles = client.noaa.list_forecast_cycles()  # combined view
print(f"NOAA/GFS — {len(noaa_cycles)} cycles available")
print("  Latest 5:", noaa_cycles[:5])

recent_cycles  = catalog.get("noaa_forecast_recent",  {}).get("cycles", [])
archive_cycles = catalog.get("noaa_forecast_archive", {}).get("cycles", [])
print(f"  Recent: {len(recent_cycles)}   Archive: {len(archive_cycles)}")

In [ ]:
# Download the most recent NOAA/GFS forecast in one call
print("Downloading latest NOAA/GFS forecast...")
path = client.noaa.download_latest(dest="./data/noaa")
print("Saved to:", path)

In [ ]:
# Download a specific cycle
if noaa_cycles:
    cycle = noaa_cycles[0]
    print(f"Downloading NOAA/GFS cycle {cycle} ...")
    paths = client.noaa.download_forecast(cycles=[cycle], dest="./data/noaa")
    print("Saved to:", paths[0])

---
## 8. Raw REST API (no SDK)

Everything above is also accessible via plain HTTP if you prefer `requests` or `curl`.

In [ ]:
import requests

API = "https://weatherdatagui-production.up.railway.app"

# --- Health ---
print(requests.get(f"{API}/api/health").json())

In [ ]:
# --- Catalog ---
catalog_raw = requests.get(f"{API}/api/catalog").json()
print(list(catalog_raw.keys()))

In [ ]:
# --- Single-file download (streamed via service account) ---
# The server proxies the file from Google Drive through a service account, so
# the response is a direct binary stream with Content-Length set — no 302
# redirects, no virus-scan confirmation pages.
# ERA5 and NOAA single files are bare .pww; HRRR single files are .zip.

quarters = catalog_raw.get("era5_na", {}).get("quarters", [])
if quarters:
    q = quarters[0]
    url = f"{API}/api/download?source=era5_na&dates={q}"
    print("Download URL:", url)
    # Uncomment to actually download:
    # resp = requests.get(url, stream=True)
    # with open(f"ERA5_NA_{q}.pww", "wb") as f:
    #     for chunk in resp.iter_content(8192):
    #         f.write(chunk)

In [ ]:
# --- Multi-file ZIP bundle ---
# Pass comma-separated date keys; server streams a ZIP.

noaa_cycles_raw = catalog_raw.get("noaa_forecast_recent", {}).get("cycles", [])
if len(noaa_cycles_raw) >= 2:
    dates_param = ",".join(noaa_cycles_raw[:2])
    url = f"{API}/api/download?source=noaa_forecast_recent&dates={dates_param}"
    print("Bundle URL:", url)
    # Uncomment to download:
    # resp = requests.get(url, stream=True)
    # with open("noaa_bundle.zip", "wb") as f:
    #     for chunk in resp.iter_content(8192):
    #         f.write(chunk)

In [ ]:
# --- Debug endpoints (useful for diagnosing missing data) ---

# Which folder IDs the catalog is using right now:
folders = requests.get(f"{API}/api/debug/folders").json()
for name, fid in folders.items():
    print(f"  {name:<28} {fid}")

In [ ]:
# Peek inside a specific folder (replace folder_id with any ID from above):
folder_id = folders.get("noaa_main", "")
if folder_id:
    result = requests.get(f"{API}/api/debug/folder", params={"folder_id": folder_id, "limit": 5}).json()
    print(f"Files in {folder_id} (first 5):")
    for f in result.get("files", []):
        print(" ", f.get("name", f))

In [ ]:
# Force a catalog refresh (bypasses the 30-minute cache):
refreshed = requests.get(f"{API}/api/catalog/refresh").json()
print("Catalog refreshed. Keys:", list(refreshed.keys()))

---
## 9. Region Catalog — Browse Available States & ISO Zones

The `/api/regions` endpoint returns every selectable region the server knows about.
Use this to find valid `region_ids` before calling the download endpoint or SDK methods.

In [ ]:
import requests

API = "https://weatherdatagui-production.up.railway.app"

regions = requests.get(f"{API}/api/regions", timeout=10).json()

states = regions["states"]
iso    = regions["iso"]

print(f"States  : {len(states)} entries")
print(f"ISO zones: {len(iso)} entries")
print()
print("First 5 states:")
for s in states[:5]:
    print(f"  {s['id']:<4}  {s['name']:<20}  bbox={s['bbox']}")

In [ ]:
# Full list of ISO / RTO zones
print("ISO / RTO zones:")
if iso:
    for z in iso:
        print(f"  {z['id']:<10}  {z['name']:<30}  bbox={z['bbox']}")
else:
    print("  (none — shapefile may not be WGS84 or path not found on server)")

In [ ]:
# Look up a specific state bbox by ID (2-letter postal code)
target = "TX"
match = next((s for s in states if s["id"] == target), None)
if match:
    print(f"{match['name']} bbox: {match['bbox']}")
    print("  Format: (lat_max, lon_min, lat_min, lon_max)")

---
## 10. Region Crop — SDK

`download_region()` is available on all three clients.
Pass either `region_ids` + `region_layer`, or a raw `bbox` tuple.

| Argument | Type | Description |
|---|---|---|
| `region_ids` | `list[str]` | State postal codes (`["TX","OK"]`) or ISO zone IDs |
| `region_layer` | `"states"` \| `"iso"` | Which layer the IDs belong to |
| `bbox` | `tuple[float,float,float,float]` | `(lat_max, lon_min, lat_min, lon_max)` |

Exactly **one** of (`region_ids` + `region_layer`) or `bbox` must be provided.

In [ ]:
# --- NOAA/GFS: crop to Texas ---
noaa_cycles = client.noaa.list_forecast_cycles()

if noaa_cycles:
    cycle = noaa_cycles[0]
    print(f"Downloading NOAA/GFS {cycle} cropped to Texas ...")
    paths = client.noaa.download_region(
        cycles=[cycle],
        region_ids=["TX"],
        region_layer="states",
        dest="./data/region/noaa",
    )
    print("Saved to:", paths[0])
else:
    print("No NOAA cycles available.")

In [ ]:
# --- NOAA/GFS: multi-state union bbox (Texas + Oklahoma + Louisiana) ---
if noaa_cycles:
    cycle = noaa_cycles[0]
    print(f"Downloading NOAA/GFS {cycle} cropped to TX + OK + LA ...")
    paths = client.noaa.download_region(
        cycles=[cycle],
        region_ids=["TX", "OK", "LA"],
        region_layer="states",
        dest="./data/region/noaa",
    )
    print("Saved to:", paths[0])

In [ ]:
# --- NOAA/GFS: custom bounding box ---
# bbox format: (lat_max, lon_min, lat_min, lon_max)
custom_bbox = (37.0, -106.5, 25.8, -93.5)   # Texas + surrounding area

if noaa_cycles:
    cycle = noaa_cycles[0]
    print(f"Downloading NOAA/GFS {cycle} with custom bbox {custom_bbox} ...")
    paths = client.noaa.download_region(
        cycles=[cycle],
        bbox=custom_bbox,
        dest="./data/region/noaa",
    )
    print("Saved to:", paths[0])

In [ ]:
# --- HRRR Forecast: crop to an ISO zone ---
forecast_cycles = client.hrrr.list_forecast_cycles()
iso_zones = regions["iso"]

if forecast_cycles and iso_zones:
    cycle = forecast_cycles[0]
    zone  = iso_zones[0]
    print(f"Downloading HRRR forecast {cycle} cropped to ISO zone '{zone['id']}' ({zone['name']}) ...")
    paths = client.hrrr.download_region(
        cycles=[cycle],
        region_ids=[zone["id"]],
        region_layer="iso",
        dest="./data/region/hrrr_forecast",
    )
    print("Saved to:", paths[0])
elif not iso_zones:
    print("No ISO zones available — shapefile may not be loaded on the server.")

In [ ]:
# --- HRRR History: crop monthly archive to Texas ---
# Requests where bbox area >= 2380 sq-deg (CONUS-scale) with the HRRR archive
# source are blocked by the server (HTTP 413) and handled locally by the SDK.

months = client.hrrr.list_months()

if months:
    month = months[0]
    print(f"Downloading HRRR history {month} cropped to Texas ...")
    paths = client.hrrr.download_region(
        months=[month],
        region_ids=["TX"],
        region_layer="states",
        dest="./data/region/hrrr_history",
    )
    print("Saved to:", paths[0])

In [ ]:
# --- ERA5: crop to Texas ---
na_quarters = client.era5.list_quarters(region="north_america")

if na_quarters:
    quarter = na_quarters[0]
    print(f"Downloading ERA5 NA {quarter} cropped to Texas ...")
    paths = client.era5.download_region(
        quarters=[quarter],
        region_ids=["TX"],
        region_layer="states",
        dest="./data/region/era5",
    )
    print("Saved to:", paths[0])

---
## 11. Region Crop — Raw REST API

Same operations as above using `requests` directly against `/api/download/region`.

**Query parameters:**

| Param | Required | Description |
|---|---|---|
| `source` | yes | Catalog source key (e.g. `noaa_forecast_recent`) |
| `dates` | yes | Comma-separated date keys |
| `region_layer` | yes | `states`, `iso`, or `custom` |
| `region_ids` | one of | Comma-separated IDs (states/iso layers) |
| `bbox` | one of | `lat_max,lon_min,lat_min,lon_max` (custom layer) |

Exactly one of `region_ids` or `bbox` must be present.

In [ ]:
import requests, os

API = "https://weatherdatagui-production.up.railway.app"

catalog_raw = requests.get(f"{API}/api/catalog").json()
noaa_cycles = catalog_raw.get("noaa_forecast_recent", {}).get("cycles", [])

if noaa_cycles:
    cycle = noaa_cycles[0]
    resp = requests.get(f"{API}/api/download/region", params={
        "source"      : "noaa_forecast_recent",
        "dates"       : cycle,
        "region_layer": "states",
        "region_ids"  : "TX",
    }, timeout=120)

    print(f"Status: {resp.status_code}  Content-Type: {resp.headers.get('Content-Type')}")

    if resp.status_code == 200:
        os.makedirs("./data/region/raw", exist_ok=True)
        fname = f"./data/region/raw/NOAA_{cycle}_TX.pww"
        with open(fname, "wb") as f:
            f.write(resp.content)
        print(f"Saved {len(resp.content):,} bytes to {fname}")
    else:
        print("Error:", resp.text[:300])

In [ ]:
# Multi-state union bbox: TX + OK + NM
if noaa_cycles:
    cycle = noaa_cycles[0]
    resp = requests.get(f"{API}/api/download/region", params={
        "source"      : "noaa_forecast_recent",
        "dates"       : cycle,
        "region_layer": "states",
        "region_ids"  : "TX,OK,NM",
    }, timeout=120)

    print(f"Status: {resp.status_code}")
    if resp.status_code == 200:
        fname = f"./data/region/raw/NOAA_{cycle}_TX_OK_NM.pww"
        with open(fname, "wb") as fh:
            fh.write(resp.content)
        print(f"Saved {len(resp.content):,} bytes to {fname}")

In [ ]:
# Custom bounding box — direct lat/lon coordinates
if noaa_cycles:
    cycle = noaa_cycles[0]
    resp = requests.get(f"{API}/api/download/region", params={
        "source"      : "noaa_forecast_recent",
        "dates"       : cycle,
        "region_layer": "custom",
        "bbox"        : "37.0,-106.5,25.8,-93.5",   # lat_max,lon_min,lat_min,lon_max
    }, timeout=120)

    print(f"Status: {resp.status_code}")
    if resp.status_code == 200:
        fname = f"./data/region/raw/NOAA_{cycle}_custom.pww"
        with open(fname, "wb") as fh:
            fh.write(resp.content)
        print(f"Saved {len(resp.content):,} bytes to {fname}")

In [ ]:
# Multi-date ZIP bundle — 2 NOAA cycles cropped to Texas
if len(noaa_cycles) >= 2:
    dates_param = ",".join(noaa_cycles[:2])
    resp = requests.get(f"{API}/api/download/region", params={
        "source"      : "noaa_forecast_recent",
        "dates"       : dates_param,
        "region_layer": "states",
        "region_ids"  : "TX",
    }, stream=True, timeout=180)

    print(f"Status: {resp.status_code}  Content-Type: {resp.headers.get('Content-Type')}")
    if resp.status_code == 200:
        fname = "./data/region/raw/NOAA_TX_bundle.zip"
        with open(fname, "wb") as fh:
            for chunk in resp.iter_content(8192):
                fh.write(chunk)
        print(f"Saved bundle to {fname}")

In [ ]:
# 413 guard — CONUS-scale bbox with HRRR archive is blocked to protect server memory
hrrr_months = catalog_raw.get("hrrr_history_archive", {}).get("months", [])

if hrrr_months:
    resp = requests.get(f"{API}/api/download/region", params={
        "source"      : "hrrr_history_archive",
        "dates"       : hrrr_months[0],
        "region_layer": "custom",
        "bbox"        : "50.0,-125.0,24.0,-66.0",   # full CONUS — blocked
    }, timeout=30)

    print(f"Status: {resp.status_code}   (expected: 413)")
    if resp.status_code == 413:
        body = resp.json()
        print("SDK hint:", body.get("sdk_hint", body))